<a href="https://www.kaggle.com/code/avikdas567/umud-challenge-muscle-architecture-ultrasound?scriptVersionId=311004530" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ================================
# UMUD Challenge - FINAL STABLE SUBMISSION (ALL FIXES)
# ================================

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import re
import warnings

warnings.filterwarnings("ignore")
os.environ["OPENCV_LOG_LEVEL"] = "ERROR"

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

# ========================
# CONFIG
# ========================
class CFG:
    IMG_SIZE = 256
    BATCH = 16
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE_PATH = "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"

APO_IMG_DIR = os.path.join(BASE_PATH, "apo_imgs_v1/apo_images_new_model_v1")
APO_MASK_DIR = os.path.join(BASE_PATH, "apo_masks_v1/apo_masks_new_model_v1")
TEST_DIR = os.path.join(BASE_PATH, "test_images_v2/test_set_v2")

# ========================
# HELPERS
# ========================
def is_image(f):
    return f.endswith((".png",".tif",".jpg"))

def get_id(name):
    nums = re.findall(r'\d+', name)
    return nums[0] if nums else name

def read_img(path):
    if path is None or not isinstance(path, str) or not os.path.exists(path):
        return np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), dtype=np.uint8)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), dtype=np.uint8)
    return img

# ========================
# BUILD TRAIN PAIRS
# ========================
img_dict = {}
mask_dict = {}

for f in os.listdir(APO_IMG_DIR):
    if is_image(f):
        img_dict[get_id(f)] = os.path.join(APO_IMG_DIR, f)

for f in os.listdir(APO_MASK_DIR):
    if is_image(f):
        mask_dict[get_id(f)] = os.path.join(APO_MASK_DIR, f)

common_ids = list(set(img_dict.keys()) & set(mask_dict.keys()))

apo_imgs = [img_dict[i] for i in common_ids]
apo_masks = [mask_dict[i] for i in common_ids]

print("✅ TRAIN SAMPLES:", len(apo_imgs))

# ========================
# DATASET
# ========================
class SegDataset(Dataset):
    def __init__(self, imgs, masks):
        self.imgs = imgs
        self.masks = masks
        self.tfms = A.Compose([
            A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
            A.Normalize(),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        img = read_img(self.imgs[i])
        mask = read_img(self.masks[i])

        mask = cv2.resize(mask, (img.shape[1], img.shape[0]))

        img = np.stack([img]*3, -1)
        mask = (mask > 0).astype(np.float32)

        data = self.tfms(image=img, mask=mask)
        return data["image"], data["mask"].unsqueeze(0)

# ========================
# MODEL
# ========================
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(3,16,3,1,1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(16,32,3,2,1), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(32,64,3,2,1), nn.ReLU())
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(64,32,2,2), nn.ReLU())
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(32,16,2,2), nn.ReLU())
        self.out = nn.Conv2d(16,1,1)

    def forward(self,x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.dec1(x)
        x = self.dec2(x)
        return self.out(x)

# ========================
# TRAIN
# ========================
train_ds = SegDataset(apo_imgs, apo_masks)
loader = DataLoader(train_ds, batch_size=CFG.BATCH, shuffle=True)

model = UNet().to(CFG.DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

print("🚀 Training...")

model.train()
for imgs, masks in loader:
    imgs = imgs.to(CFG.DEVICE)
    masks = masks.to(CFG.DEVICE)

    preds = model(imgs)
    loss = loss_fn(preds, masks)

    opt.zero_grad()
    loss.backward()
    opt.step()

print("✅ Training complete")

# ========================
# LOAD TEST (FIXED MAPPING)
# ========================
sample_sub = pd.read_csv(os.path.join(BASE_PATH, "sample_submission.csv"), sep=";")
id_col = [c for c in sample_sub.columns if "id" in c.lower()][0]

test_dict = {}
for f in os.listdir(TEST_DIR):
    if is_image(f):
        key = get_id(f)
        test_dict[key] = os.path.join(TEST_DIR, f)

def get_path(img_id):
    img_id = get_id(str(img_id))
    return test_dict.get(img_id, list(test_dict.values())[0])

sample_sub["path"] = sample_sub[id_col].apply(get_path)

print("Missing paths:", sample_sub["path"].isnull().sum())

# ========================
# GEOMETRY
# ========================
def compute_mt(mask):
    ys = np.where(mask>0)[0]
    return np.percentile(ys,95) - np.percentile(ys,5) if len(ys)>10 else 20

def compute_fl(mask):
    ys,xs = np.where(mask>0)
    return np.sqrt((xs.max()-xs.min())**2+(ys.max()-ys.min())**2) if len(xs)>10 else 80

def compute_pa(mask):
    ys,xs = np.where(mask>0)
    if len(xs)<10: return 15
    [vx,vy,x0,y0] = cv2.fitLine(np.column_stack([xs,ys]).astype(np.float32),
                               cv2.DIST_L2,0,0.01,0.01)
    return abs(np.degrees(np.arctan2(vy,vx)))

# ========================
# INFERENCE
# ========================
model.eval()

pa_list, fl_list, mt_list = [], [], []

for path in tqdm(sample_sub["path"]):
    img = read_img(path)
    img = cv2.resize(img,(CFG.IMG_SIZE,CFG.IMG_SIZE))

    inp = torch.tensor(img).float().unsqueeze(0).unsqueeze(0)/255
    inp = inp.repeat(1,3,1,1).to(CFG.DEVICE)

    with torch.no_grad():
        pred = torch.sigmoid(model(inp)).cpu().numpy()[0,0]

    mask = (pred > 0.5).astype(np.uint8)

    pa_list.append(compute_pa(mask))
    fl_list.append(compute_fl(mask))
    mt_list.append(compute_mt(mask))

# ========================
# SAVE
# ========================
sample_sub["pa_deg"] = np.clip(pa_list,5,45)
sample_sub["fl_mm"] = np.clip(fl_list,30,200)
sample_sub["mt_mm"] = np.clip(mt_list,10,50)

sample_sub.to_csv("submission.csv", index=False)

print("🏆 FINAL SUBMISSION READY")

✅ TRAIN SAMPLES: 1048
🚀 Training...


[ WARN:0@45.389] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@45.389] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@45.892] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@45.892] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@46.219] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@46.219] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@47.663] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@47.663] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered


✅ Training complete
Missing paths: 0


100%|██████████| 2/2 [00:00<00:00, 27.56it/s]

🏆 FINAL SUBMISSION READY
